In [2]:
# =====================================
# RANDOM FOREST
# =====================================

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# =====================================
# LOAD DATA
# =====================================

df = pd.read_excel("final_filled_climate_cleaned.xlsx")

df["Date"] = pd.to_datetime(
    df["Date"],
    dayfirst=True,
    errors="coerce"
)

df = df.sort_values(
    ["Station", "Date"]
).reset_index(drop=True)

# =====================================
# TIME FEATURES
# =====================================

df["Month"] = df["Date"].dt.month
df["Week"] = df["Date"].dt.isocalendar().week.astype(int)
df["DayOfWeek"] = df["Date"].dt.dayofweek
df["Year"] = df["Date"].dt.year

# =====================================
# STATION ENCODING
# =====================================

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["Station_enc"] = le.fit_transform(
    df["Station"]
)

# =====================================
# CREATE LAG FEATURES
# =====================================

df["Temp_lag1"] = df.groupby("Station")["Temp"].shift(1)
df["Temp_lag7"] = df.groupby("Station")["Temp"].shift(7)

df["RH_lag1"] = df.groupby("Station")["RH"].shift(1)
df["RH_lag7"] = df.groupby("Station")["RH"].shift(7)

df["Rain_lag1"] = df.groupby("Station")["Rain"].shift(1)
df["Rain_lag7"] = df.groupby("Station")["Rain"].shift(7)

df["PRESS_lag1"] = df.groupby("Station")["PRESS"].shift(1)
df["PRESS_lag7"] = df.groupby("Station")["PRESS"].shift(7)

df["SUNSHINE_lag1"] = df.groupby("Station")["SUNSHINE"].shift(1)
df["SUNSHINE_lag7"] = df.groupby("Station")["SUNSHINE"].shift(7)

# remove first few rows created by lagging
df = df.dropna().reset_index(drop=True)

# =====================================
# FEATURES
# =====================================

features = [

    "Temp_lag1",
    "Temp_lag7",

    "RH_lag1",
    "RH_lag7",

    "Rain_lag1",
    "Rain_lag7",

    "PRESS_lag1",
    "PRESS_lag7",

    "SUNSHINE_lag1",
    "SUNSHINE_lag7",

    "Month",
    "Week",
    "DayOfWeek",

    "Station_enc"
]

targets = [
    "Temp",
    "RH",
    "Rain",
    "PRESS",
    "SUNSHINE"
]

# =====================================
# CHRONOLOGICAL SPLIT
# =====================================

train_df = df[df["Year"] <= 2022]
test_df = df[df["Year"] == 2024]

X_train = train_df[features]
Y_train = train_df[targets]

X_test = test_df[features]
Y_test = test_df[targets]

# =====================================
# RANDOM FOREST
# =====================================

rf = MultiOutputRegressor(

    RandomForestRegressor(
        n_estimators=300,
        max_depth=20,
        random_state=42,
        n_jobs=-1
    )

)

rf.fit(
    X_train,
    Y_train
)

# =====================================
# PREDICTION
# =====================================

Y_pred = rf.predict(X_test)

# =====================================
# EVALUATION
# =====================================

results = []

for i, target in enumerate(targets):

    mae = mean_absolute_error(
        Y_test.iloc[:, i],
        Y_pred[:, i]
    )

    mse = mean_squared_error(
        Y_test.iloc[:, i],
        Y_pred[:, i]
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        Y_test.iloc[:, i],
        Y_pred[:, i]
    )

    results.append({

        "Target": target,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2

    })

results_df = pd.DataFrame(results)

print(results_df)

results_df.to_excel(
    "RandomForest_Results.xlsx",
    index=False
)

print(
    "Saved: RandomForest_Results.xlsx"
)

     Target       MAE         MSE       RMSE        R2
0      Temp  1.200325    2.755661   1.660019  0.946033
1        RH  8.437128  129.918988  11.398201  0.655199
2      Rain  1.807485   21.289971   4.614106  0.205155
3     PRESS  1.519946    3.824873   1.955728  0.998682
4  SUNSHINE  2.111612   10.703820   3.271669  0.418375
Saved: RandomForest_Results.xlsx


In [3]:
RandomForestRegressor(
    n_estimators=500,
    max_depth=30,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",30
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max

In [4]:
# =====================================================
# XGBOOST BASELINE FOR ALL TARGETS AND STATIONS
# =====================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor


# =====================================================
# 1. LOAD DATA
# =====================================================

df = pd.read_excel("final_filled_climate_cleaned.xlsx")

df["Date"] = pd.to_datetime(
    df["Date"],
    dayfirst=True,
    errors="coerce"
)

df = df.dropna(subset=["Date"])
df = df.sort_values(["Station", "Date"]).reset_index(drop=True)


# =====================================================
# 2. TIME FEATURES
# =====================================================

df["Month"] = df["Date"].dt.month
df["Week"] = df["Date"].dt.isocalendar().week.astype(int)
df["DayOfWeek"] = df["Date"].dt.dayofweek
df["Year"] = df["Date"].dt.year


# =====================================================
# 3. STATION ENCODING
# =====================================================

le = LabelEncoder()
df["Station_enc"] = le.fit_transform(df["Station"])


# =====================================================
# 4. CREATE LAG FEATURES
# =====================================================

for col in ["Temp", "RH", "Wind", "Rain", "PRESS", "SUNSHINE"]:
    df[f"{col}_lag1"] = df.groupby("Station")[col].shift(1)
    df[f"{col}_lag7"] = df.groupby("Station")[col].shift(7)
    df[f"{col}_lag30"] = df.groupby("Station")[col].shift(30)

df = df.dropna().reset_index(drop=True)


# =====================================================
# 5. FEATURES AND TARGETS
# =====================================================

features = [
    "Temp_lag1", "Temp_lag7", "Temp_lag30",
    "RH_lag1", "RH_lag7", "RH_lag30",
    "Wind_lag1", "Wind_lag7", "Wind_lag30",
    "Rain_lag1", "Rain_lag7", "Rain_lag30",
    "PRESS_lag1", "PRESS_lag7", "PRESS_lag30",
    "SUNSHINE_lag1", "SUNSHINE_lag7", "SUNSHINE_lag30",
    "Month", "Week", "DayOfWeek",
    "Station_enc"
]

targets = [
    "Temp",
    "RH",
    "Rain",
    "PRESS",
    "SUNSHINE"
]


# =====================================================
# 6. CHRONOLOGICAL SPLIT
# =====================================================

train_df = df[df["Year"] <= 2022].copy()
test_df  = df[df["Year"] == 2024].copy()

X_train = train_df[features]
Y_train = train_df[targets]

X_test = test_df[features]
Y_test = test_df[targets]


# =====================================================
# 7. XGBOOST MODEL
# =====================================================

xgb = MultiOutputRegressor(
    XGBRegressor(
        n_estimators=500,
        max_depth=5,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )
)

xgb.fit(X_train, Y_train)


# =====================================================
# 8. PREDICTION
# =====================================================

Y_pred = xgb.predict(X_test)


# =====================================================
# 9. EVALUATION
# =====================================================

results = []

for i, target in enumerate(targets):

    mae = mean_absolute_error(Y_test.iloc[:, i], Y_pred[:, i])
    mse = mean_squared_error(Y_test.iloc[:, i], Y_pred[:, i])
    rmse = np.sqrt(mse)
    r2 = r2_score(Y_test.iloc[:, i], Y_pred[:, i])

    results.append({
        "Target": target,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(results)

print(results_df)

results_df.to_excel(
    "XGBoost_Results.xlsx",
    index=False
)

print("Saved: XGBoost_Results.xlsx")

     Target       MAE         MSE       RMSE        R2
0      Temp  1.144208    2.425174   1.557297  0.952505
1        RH  8.296817  124.288440  11.148473  0.670142
2      Rain  1.689573   21.583598   4.645815  0.194193
3     PRESS  1.552376    3.995404   1.998851  0.998623
4  SUNSHINE  1.674222    6.012714   2.452084  0.673281
Saved: XGBoost_Results.xlsx
